In [1]:
#Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
#Import Libraries
import os
import pandas as pd
import numpy as np

In [3]:
## File paths
base_path = "/content/drive/MyDrive/IIT/DSPL/ICW/data/"

df_main = pd.read_csv(base_path + "API_IT.NET.USER.csv")
df_country = pd.read_csv(base_path + "Metadata_Country_API_IT.NET.USER.csv")
df_indicator = pd.read_csv(base_path + "Metadata_Indicator_API_IT.NET.USER.csv")

# Quick check
df_main.head()

,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,Aruba,ABW,Individuals using the Internet (% of population),IT.NET.USER.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,93.542454,97.17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Africa Eastern and Southern,AFE,Individuals using the Internet (% of population),IT.NET.USER.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,16.300000,17.30,19.600000,21.600000,23.5000,25.000000,26.800000,27.800000,28.800000,30.4
2,Afghanistan,AFG,Individuals using the Internet (% of population),IT.NET.USER.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,11.000000,13.50,16.799999,17.600000,17.0485,16.514299,15.866300,15.928400,16.094999,NaN
3,Africa Western and Central,AFW,Individuals using the Internet (% of population),IT.NET.USER.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,20.900000,23.80,25.700000,28.500000,31.5000,34.700000,37.500000,38.500000,40.000000,42.4
4,Angola,AGO,Individuals using the Internet (% of population),IT.NET.USER.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,23.200001,26.00,29.000000,32.129398,33.3209,34.556499,35.838001,37.885899,40.704800,NaN


In [4]:
# Keep identifier columns
id_cols = ["Country Name", "Country Code", "Indicator Name", "Indicator Code"]

# Convert to long format
df_long = df_main.melt(
    id_vars=id_cols,
    var_name="Year",
    value_name="Internet_Usage"
)

df_long.head()

,Country Name,Country Code,Indicator Name,Indicator Code,Year,Internet_Usage
0,Aruba,ABW,Individuals using the Internet (% of population),IT.NET.USER.ZS,1960,NaN
1,Africa Eastern and Southern,AFE,Individuals using the Internet (% of population),IT.NET.USER.ZS,1960,NaN
2,Afghanistan,AFG,Individuals using the Internet (% of population),IT.NET.USER.ZS,1960,NaN
3,Africa Western and Central,AFW,Individuals using the Internet (% of population),IT.NET.USER.ZS,1960,NaN
4,Angola,AGO,Individuals using the Internet (% of population),IT.NET.USER.ZS,1960,NaN


In [5]:
# Convert Year to numeric
df_long["Year"] = pd.to_numeric(df_long["Year"], errors="coerce")

# Convert Internet usage to numeric
df_long["Internet_Usage"] = pd.to_numeric(df_long["Internet_Usage"], errors="coerce")

# Drop missing values
df_long = df_long.dropna(subset=["Internet_Usage"])

# Filter meaningful years (Internet starts ~1990)
df_long = df_long[df_long["Year"] >= 1990]

df_long.head()

,Country Name,Country Code,Indicator Name,Indicator Code,Year,Internet_Usage
7980,Aruba,ABW,Individuals using the Internet (% of population),IT.NET.USER.ZS,1990,0.0
7982,Afghanistan,AFG,Individuals using the Internet (% of population),IT.NET.USER.ZS,1990,0.0
7984,Angola,AGO,Individuals using the Internet (% of population),IT.NET.USER.ZS,1990,0.0
7985,Albania,ALB,Individuals using the Internet (% of population),IT.NET.USER.ZS,1990,0.0
7986,Andorra,AND,Individuals using the Internet (% of population),IT.NET.USER.ZS,1990,0.0


In [6]:
# Merge region & income group
df_final = df_long.merge(
    df_country,
    on="Country Code",
    how="left"
)

df_final.head()

,Country Name,Country Code,Indicator Name,Indicator Code,Year,Internet_Usage,Region,IncomeGroup,SpecialNotes,TableName,Unnamed: 5
0,Aruba,ABW,Individuals using the Internet (% of population),IT.NET.USER.ZS,1990,0.0,Latin America & Caribbean,High income,NaN,Aruba,NaN
1,Afghanistan,AFG,Individuals using the Internet (% of population),IT.NET.USER.ZS,1990,0.0,Middle East & North Africa,Low income,The reporting period for national accounts dat...,Afghanistan,NaN
2,Angola,AGO,Individuals using the Internet (% of population),IT.NET.USER.ZS,1990,0.0,Sub-Saharan Africa,Lower middle income,The World Bank systematically assesses the app...,Angola,NaN
3,Albania,ALB,Individuals using the Internet (% of population),IT.NET.USER.ZS,1990,0.0,Europe & Central Asia,Upper middle income,NaN,Albania,NaN
4,Andorra,AND,Individuals using the Internet (% of population),IT.NET.USER.ZS,1990,0.0,Europe & Central Asia,High income,NaN,Andorra,NaN


In [7]:
#Remove Aggregated Rows
exclude_regions = ["Aggregates"]

df_final = df_final[~df_final["Region"].isna()]

In [8]:
print("Shape:", df_final.shape)
print(df_final.columns)

df_final.describe()

Shape: (6224, 11)
Index(['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code',
       'Year', 'Internet_Usage', 'Region', 'IncomeGroup', 'SpecialNotes',
       'TableName', 'Unnamed: 5'],
      dtype='object')


,Year,Internet_Usage,Unnamed: 5
count,6224.000000,6224.000000,0.0
mean,2008.291292,32.000600,NaN
std,9.395320,32.674237,NaN
min,1990.000000,0.000000,NaN
25%,2001.000000,1.800000,NaN
50%,2008.000000,18.965050,NaN
75%,2016.000000,61.957967,NaN
max,2025.000000,100.000000,NaN


In [15]:
# Drop unnecessary columns
df_final = df_final.drop(columns=["SpecialNotes", "TableName", "Unnamed: 5","Indicator Name","Indicator Code"], errors='ignore')

df_final.head()

,Country,Code,Year,Internet_Users_Percent,Region,IncomeGroup
0,Aruba,ABW,1990,0.0,Latin America & Caribbean,High income
1,Afghanistan,AFG,1990,0.0,Middle East & North Africa,Low income
2,Angola,AGO,1990,0.0,Sub-Saharan Africa,Lower middle income
3,Albania,ALB,1990,0.0,Europe & Central Asia,Upper middle income
4,Andorra,AND,1990,0.0,Europe & Central Asia,High income


In [16]:
#Rename columns
df_final = df_final.rename(columns={
    "Country Name": "Country",
    "Country Code": "Code",
    "Internet_Usage": "Internet_Users_Percent"
})

In [17]:
#SAVE CLEAN DATA
output_path = "/content/drive/MyDrive/IIT/DSPL/ICW/data/processed/cleaned_data.csv"
df_final.to_csv(output_path, index=False)

print("Clean dataset saved at:", output_path)

Clean dataset saved at: /content/drive/MyDrive/IIT/DSPL/ICW/data/processed/cleaned_data.csv
